In [1]:
import numpy as np 
import pandas as pd
mens_teams = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MTeams.csv")
seeds_men = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MNCAATourneySeeds.csv")
mens_tourney_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv")
mens_season_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MRegularSeasonCompactResults.csv")


womens_teams = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/WTeams.csv")
seeds_women = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/WNCAATourneySeeds.csv")
womens_tourney_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/WNCAATourneyCompactResults.csv")
womens_season_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv")

massey_ordinals =pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MMasseyOrdinals.csv") #tbh idk what massey even is.. i dont speak basketballenese
stage_1_submission = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage1.csv")
stage_2_submission = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage2.csv")


In [2]:
stage_1_submission.head()

,ID,Pred
0,2022_1101_1102,0.5
1,2022_1101_1103,0.5
2,2022_1101_1104,0.5
3,2022_1101_1105,0.5
4,2022_1101_1106,0.5


In [3]:
from pathlib import Path


STAGE = 1 #choose the stage submission as 2 for final submission :D
OUT_DIR = Path("/kaggle/working")

def parse_seed(s):
    if pd.isna(s):
        return np.nan
    return int("".join([c for c in str(s) if c.isdigit()]))

if "SeedNum" not in seeds_men.columns:
    seeds_men["SeedNum"] = seeds_men["Seed"].apply(parse_seed)
if "SeedNum" not in seeds_women.columns:
    seeds_women["SeedNum"] = seeds_women["Seed"].apply(parse_seed)

def build_season_stats(reg_df):
    w = reg_df[["Season","WTeamID","WScore","LScore"]].copy()
    w.columns = ["Season","TeamID","ScoreFor","ScoreAgainst"]
    w["Win"] = 1
    l = reg_df[["Season","LTeamID","LScore","WScore"]].copy()
    l.columns = ["Season","TeamID","ScoreFor","ScoreAgainst"]
    l["Win"] = 0
    df = pd.concat([w, l], ignore_index=True)
    df["Margin"] = df["ScoreFor"] - df["ScoreAgainst"]
    stats = df.groupby(["Season","TeamID"]).agg(
        Games=("Win","count"),
        Wins=("Win","sum"),
        AvgScore=("ScoreFor","mean"),
        AvgMargin=("Margin","mean")
    ).reset_index()
    stats["WinRate"] = stats["Wins"] / stats["Games"]
    return stats

m_stats = build_season_stats(mens_season_results)
w_stats = build_season_stats(womens_season_results)

massey_mor = (
    massey_ordinals[massey_ordinals["SystemName"] == "MOR"]
    .sort_values(["Season","RankingDayNum"])
    .groupby(["Season","TeamID"]).last()
    .reset_index()[["Season","TeamID","OrdinalRank"]]
    .rename(columns={"OrdinalRank":"MOR"})
)

def build_team_features(stats, seeds, mor_df=None):
    df = stats.merge(seeds[["Season","TeamID","SeedNum"]], on=["Season","TeamID"], how="left")
    if mor_df is not None:
        df = df.merge(mor_df, on=["Season","TeamID"], how="left")
    else:
        df["MOR"] = np.nan
    return df

m_feats = build_team_features(m_stats, seeds_men, massey_mor)
w_feats = build_team_features(w_stats, seeds_women, None)

def build_matchup_df(df_ids, team_feats, is_men, has_label):
    df = df_ids.copy()
    if has_label:
        df["Label"] = df["Label"].astype(int)
    df["is_men"] = is_men

    df = df.merge(team_feats, left_on=["Season","T1"], right_on=["Season","TeamID"], how="left")
    df = df.merge(team_feats, left_on=["Season","T2"], right_on=["Season","TeamID"], how="left", suffixes=("_T1","_T2"))
    df = df.drop(columns=["TeamID_T1","TeamID_T2"])

    for col in ["WinRate","AvgMargin","AvgScore","SeedNum","MOR","Games","Wins"]:
        if f"{col}_T1" in df.columns and f"{col}_T2" in df.columns:
            df[f"{col}Diff"] = df[f"{col}_T1"] - df[f"{col}_T2"]

    return df

def build_train_from_tourney(tourney, team_feats, is_men):
    t1 = np.minimum(tourney["WTeamID"], tourney["LTeamID"])
    t2 = np.maximum(tourney["WTeamID"], tourney["LTeamID"])
    label = (tourney["WTeamID"] == t1).astype(int)
    df_ids = pd.DataFrame({"Season": tourney["Season"], "T1": t1, "T2": t2, "Label": label})
    return build_matchup_df(df_ids, team_feats, is_men, has_label=True)

train_m = build_train_from_tourney(mens_tourney_results, m_feats, 1)
train_w = build_train_from_tourney(womens_tourney_results, w_feats, 0)
train_df = pd.concat([train_m, train_w], ignore_index=True)

sub = (stage_1_submission if STAGE == 1 else stage_2_submission).copy()
sub[["Season","T1","T2"]] = sub["ID"].str.split("_", expand=True).astype(int)

m_team_ids = set(mens_teams["TeamID"])
sub["Gender"] = sub["T1"].apply(lambda x: "M" if x in m_team_ids else "W")

test_m = build_matchup_df(sub[sub["Gender"]=="M"][["ID","Season","T1","T2"]], m_feats, 1, has_label=False)
test_w = build_matchup_df(sub[sub["Gender"]=="W"][["ID","Season","T1","T2"]], w_feats, 0, has_label=False)
test_df = pd.concat([test_m, test_w], ignore_index=True)

OUT_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(OUT_DIR / "train.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)

print(f"Wrote {OUT_DIR / 'train.csv'} ({len(train_df):,} rows)")
print(f"Wrote {OUT_DIR / 'test.csv'} ({len(test_df):,} rows)")


Wrote /kaggle/working/train.csv (4,302 rows)
Wrote /kaggle/working/test.csv (519,144 rows)


In [4]:
train_df.head(10)

,Season,T1,T2,Label,is_men,Games_T1,Wins_T1,AvgScore_T1,AvgMargin_T1,WinRate_T1,...,WinRate_T2,SeedNum_T2,MOR_T2,WinRateDiff,AvgMarginDiff,AvgScoreDiff,SeedNumDiff,MORDiff,GamesDiff,WinsDiff
0,1985,1116,1234,1,1,33,21,65.333333,3.636364,0.636364,...,0.666667,8.0,NaN,-0.030303,-6.830303,-4.400000,1.0,NaN,3,1
1,1985,1120,1345,1,1,29,18,70.344828,3.689655,0.620690,...,0.680000,6.0,NaN,-0.059310,-0.110345,1.224828,5.0,NaN,4,1
2,1985,1207,1250,1,1,27,25,75.740741,15.666667,0.925926,...,0.379310,16.0,NaN,0.546616,20.114943,9.982120,-15.0,NaN,-2,14
3,1985,1229,1425,1,1,27,20,71.592593,5.962963,0.740741,...,0.678571,8.0,NaN,0.062169,2.177249,3.199735,1.0,NaN,-1,1
4,1985,1242,1325,1,1,30,23,76.033333,5.633333,0.766667,...,0.740741,14.0,NaN,0.025926,1.077778,8.477778,-11.0,NaN,3,3
5,1985,1246,1449,1,1,28,16,65.607143,2.357143,0.571429,...,0.709677,5.0,NaN,-0.138249,-4.191244,0.478111,7.0,NaN,-3,-6
6,1985,1256,1338,1,1,28,26,77.500000,12.678571,0.928571,...,0.576923,12.0,NaN,0.351648,10.101648,7.269231,-7.0,NaN,2,11
7,1985,1233,1260,0,1,29,25,74.448276,7.344828,0.862069,...,0.833333,4.0,NaN,0.028736,0.578161,-11.385057,9.0,NaN,-1,0
8,1985,1292,1314,0,1,26,13,67.230769,0.115385,0.500000,...,0.741935,2.0,NaN,-0.241935,-7.529777,-7.511166,13.0,NaN,-5,-10
9,1985,1323,1333,1,1,28,20,69.857143,7.571429,0.714286,...,0.724138,10.0,NaN,-0.009852,2.916256,7.960591,-3.0,NaN,-1,-1


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
import xgboost as xgb
import inspect

OUT_DIR = Path("/kaggle/working")
train_df = pd.read_csv(OUT_DIR / "train.csv")
test_df = pd.read_csv(OUT_DIR / "test.csv")

target_col = "Label"
drop_cols = {"Label","ID"}
feature_cols = [c for c in train_df.columns if c not in drop_cols]

train_mask = train_df["Season"] < 2025
val_mask = train_df["Season"] == 2025

X_train = train_df.loc[train_mask, feature_cols]
y_train = train_df.loc[train_mask, target_col].astype(int)

X_val = train_df.loc[val_mask, feature_cols]
y_val = train_df.loc[val_mask, target_col].astype(int)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "learning_rate": 0.002,
    "max_depth": 15,
    "min_child_weight": 3,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "seed": 42
}

def brier_metric(preds, dmatrix):
    y_true = dmatrix.get_label()
    brier = float(np.mean((preds - y_true) ** 2))
    return "brier", brier

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

train_kwargs = dict(
    params=params,
    dtrain=dtrain,
    num_boost_round=5000,
    evals=[(dval, "val")],
    early_stopping_rounds=300,
    verbose_eval=50
)

sig = inspect.signature(xgb.train)
if "custom_metric" in sig.parameters:
    train_kwargs["custom_metric"] = brier_metric
else:
    train_kwargs["feval"] = brier_metric

model = xgb.train(**train_kwargs)

val_pred = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
val_ll = log_loss(y_val, val_pred)
val_brier = np.mean((val_pred - y_val) ** 2)
print(f"Validation (2025) LogLoss: {val_ll:.6f}")
print(f"Validation (2025) Brier:   {val_brier:.6f}")

X_full = train_df[feature_cols]
y_full = train_df[target_col].astype(int)

dfull = xgb.DMatrix(X_full, label=y_full)
model_full = xgb.train(
    params,
    dfull,
    num_boost_round=(model.best_iteration + 1)
)

dtest = xgb.DMatrix(test_df[feature_cols])
test_pred = model_full.predict(dtest)

sub = pd.DataFrame({"ID": test_df["ID"].values, "Pred": test_pred})
sub["Pred"] = sub["Pred"].clip(0.0001, 0.9999)

sub_path = OUT_DIR / "submission.csv"
sub.to_csv(sub_path, index=False)
print(f"Wrote {sub_path}")


[0]	val-logloss:0.69194	val-brier:0.24940
[50]	val-logloss:0.66025	val-brier:0.23358
[100]	val-logloss:0.63319	val-brier:0.22020
[150]	val-logloss:0.60969	val-brier:0.20876
[200]	val-logloss:0.58926	val-brier:0.19902
[250]	val-logloss:0.57146	val-brier:0.19074
[300]	val-logloss:0.55492	val-brier:0.18321
[350]	val-logloss:0.54112	val-brier:0.17717
[400]	val-logloss:0.52856	val-brier:0.17181
[450]	val-logloss:0.51741	val-brier:0.16721
[500]	val-logloss:0.50720	val-brier:0.16309
[550]	val-logloss:0.49843	val-brier:0.15967
[600]	val-logloss:0.48996	val-brier:0.15646
[650]	val-logloss:0.48297	val-brier:0.15391
[700]	val-logloss:0.47628	val-brier:0.15149
[750]	val-logloss:0.47110	val-brier:0.14980
[800]	val-logloss:0.46565	val-brier:0.14793
[850]	val-logloss:0.46062	val-brier:0.14625
[900]	val-logloss:0.45598	val-brier:0.14473
[950]	val-logloss:0.45172	val-brier:0.14332
[1000]	val-logloss:0.44779	val-brier:0.14206
[1050]	val-logloss:0.44453	val-brier:0.14108
[1100]	val-logloss:0.44166	val-br